In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/laptop_cleaned.csv")

In [2]:
print("Shape:", df.shape)

Shape: (8176, 67)


In [3]:
print("Number of columns:", len(df.columns))
print(df.columns.tolist())

Number of columns: 67
['Brand', 'Model', 'Series', 'Thickness', 'Dimensions (WxDxH)', 'Weight', 'Colors', 'Operating System', 'Operating System Type', 'Display Size', 'Display Resolution', 'Pixel Density', 'Display Type', 'Display Features', 'Display Touchscreen', 'Processor', 'Clock-speed', 'Chipset', 'Cache', 'Graphic Processor', 'Capacity', 'RAM Type', 'RAM Speed', 'Memory Slots', 'Memory Layout', 'SSD Capacity', 'SSD Type', 'Battery Cell', 'Battery Type', 'Power Supply', 'Wireless LAN', 'Wi-Fi Version', 'Bluetooth', 'Bluetooth Version', 'HDMI Ports', 'USB 2.0 slots', 'SD Card Reader', 'Ethernet ports', 'Headphone Jack', 'Microphone Jack', 'Web-cam', 'Video Recording', 'Speakers', 'In-built Microphone', 'Microphone Type', 'Pointing device', 'Keyboard', 'Fingerprint scanner', 'Warranty', 'Sales Package', 'Refresh Rate', 'Brightness', 'Graphics Memory', 'Expandable Memory', 'Backlit Keyboard', 'HDD Capacity', 'HDD Speed(RPM)', 'HDD Type', 'USB 3.0 slots', 'Battery Life', 'USB Type C',

In [4]:
target = "Price (Rs)"
print("Target dtype:", df[target].dtype)
print("Missing target values:", df[target].isna().sum())

Target dtype: float64
Missing target values: 0


In [5]:
print(df.dtypes.value_counts())

str        41
float64    26
Name: count, dtype: int64


In [6]:
missing = df.isna().sum()

missing = missing[missing > 0].sort_values(ascending=False)

print(missing)
print("no of columns with missing values",len(missing))

Chipset                  7133
Aspect Ratio             6849
HDD Type                 6448
HDD Speed(RPM)           6440
HDD Capacity             6323
SSD Interface            6128
USB 3.0 slots            5955
Brightness               5807
Battery Capacity         5732
SSD Type                 5729
Battery Life             5457
Number of Cores          5259
Graphics Memory          5111
Refresh Rate             5069
USB 2.0 slots            4952
Expandable Memory        4948
Ethernet ports           4741
Display Type             3876
SD Card Reader           3700
Backlit Keyboard         3590
Operating System Type    3504
RAM Speed                3082
USB Type C               3005
Fingerprint scanner      2674
Pointing device          2311
Cache                    1560
Power Supply             1414
SSD Capacity             1333
Battery Cell             1300
Video Recording          1151
Series                   1090
Keyboard                 1074
Bluetooth Version         815
HDMI Ports

In [7]:
duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


# Complete Feature Inventory

In [8]:
feature_inventory = pd.DataFrame({
    "Feature": df.columns,
    "Data Type": df.dtypes.astype(str),
    "Unique Values": df.nunique(dropna=True),
    "Missing Count": df.isna().sum(),
    "Missing %": (df.isna().mean() * 100).round(2)
})

feature_inventory = feature_inventory[
    feature_inventory["Feature"] != "Price (Rs)"
].reset_index(drop=True)

print("Number of candidate features:", len(feature_inventory))
feature_inventory

Number of candidate features: 66


,Feature,Data Type,Unique Values,Missing Count,Missing %
0,Brand,str,35,0,0.00
1,Model,str,8116,0,0.00
2,Series,str,532,1090,13.33
3,Thickness,float64,256,678,8.29
4,Dimensions (WxDxH),str,2274,680,8.32
...,...,...,...,...,...
61,Battery Capacity,float64,89,5732,70.11
62,Aspect Ratio,str,3,6849,83.77
63,Number of Cores,float64,12,5259,64.32
64,SSD Interface,str,4,6128,74.95


In [9]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

display(feature_inventory)

,Feature,Data Type,Unique Values,Missing Count,Missing %
0,Brand,str,35,0,0.00
1,Model,str,8116,0,0.00
2,Series,str,532,1090,13.33
3,Thickness,float64,256,678,8.29
4,Dimensions (WxDxH),str,2274,680,8.32
5,Weight,float64,228,306,3.74
6,Colors,str,470,176,2.15
7,Operating System,str,32,0,0.00
8,Operating System Type,str,2,3504,42.86
9,Display Size,float64,37,3,0.04


# feature engineering

In [10]:
print(df["Capacity"].value_counts(dropna=False))

Capacity
8 GB      3301
16 GB     3122
4 GB      1159
32 GB      421
24 GB       57
2 GB        38
64 GB       29
12 GB       22
36 GB       10
48 GB        6
18 GB        6
3 GB         2
6 GB         2
128 GB       1
Name: count, dtype: int64


In [11]:
print(
    df["Capacity"]
    .astype(str)
    .str.match(r"^\d+(\.\d+)?\s*GB$", case=False)
    .value_counts()
)

Capacity
True    8176
Name: count, dtype: int64


In [12]:
feature_decisions = {

    # ============================================================
    # Group 1 — Identity / administrative
    # ============================================================

    "Brand": "KEEP",
    "Model": "REMOVE",
    "Series": "KEEP",
    "Warranty": "REMOVE",
    "Sales Package": "REMOVE",
    "market_status": "REMOVE",

    # ============================================================
    # Group 2 — Physical characteristics
    # ============================================================

    "Thickness": "KEEP",
    "Dimensions (WxDxH)": "REMOVE",
    "Weight": "KEEP",
    "Colors": "REMOVE",

    # ============================================================
    # Group 3 — Operating system
    # ============================================================

    "Operating System": "KEEP",
    "Operating System Type": "REMOVE",

    # ============================================================
    # Group 4 — Display
    # ============================================================

    "Display Size": "KEEP",
    "Display Resolution": "REMOVE",
    "Pixel Density": "REMOVE",
    "Display Type": "REMOVE",
    "Display Features": "REMOVE",
    "Display Touchscreen": "KEEP",

    # ============================================================
    # Group 5 — Processor
    # ============================================================

    "Processor": "KEEP",
    "Clock-speed": "REMOVE",
    "Chipset": "REMOVE",
    "Cache": "REMOVE",
    "Number of Cores": "REMOVE",

    # ============================================================
    # Group 6 — GPU
    # ============================================================

    "Graphic Processor": "KEEP",
    "Graphics Memory": "REMOVE",

    # ============================================================
    # Group 7 — RAM
    # ============================================================

    "Capacity": "TRANSFORM",
    "RAM Type": "KEEP",
    "RAM Speed": "REMOVE",
    "Memory Slots": "REMOVE",
    "Memory Layout": "REMOVE",
    "Expandable Memory": "REMOVE",

    # ============================================================
    # Group 8 — Storage
    # ============================================================

    "SSD Capacity": "KEEP",
    "SSD Type": "REMOVE",
    "SSD Interface": "REMOVE",
    "HDD Capacity": "KEEP",
    "HDD Speed(RPM)": "REMOVE",
    "HDD Type": "REMOVE",

    # ============================================================
    # Group 9 — Battery
    # ============================================================

    "Battery Cell": "REMOVE",
    "Battery Type": "REMOVE",
    "Battery Life": "REMOVE",
    "Battery Capacity": "KEEP",

    # ============================================================
    # Group 10 — Connectivity / ports
    # ============================================================

    "Wireless LAN": "REMOVE",
    "Wi-Fi Version": "REMOVE",
    "Bluetooth": "REMOVE",
    "Bluetooth Version": "REMOVE",
    "HDMI Ports": "REMOVE",
    "USB 2.0 slots": "REMOVE",
    "USB 3.0 slots": "REMOVE",
    "USB Type C": "REMOVE",
    "Ethernet ports": "REMOVE",
    "SD Card Reader": "REMOVE",
    "Headphone Jack": "REMOVE",
    "Microphone Jack": "REMOVE",

    # ============================================================
    # Group 11 — Camera / audio / input
    # ============================================================

    "Web-cam": "REMOVE",
    "Video Recording": "REMOVE",
    "Speakers": "REMOVE",
    "In-built Microphone": "REMOVE",
    "Microphone Type": "REMOVE",
    "Pointing device": "REMOVE",
    "Keyboard": "REMOVE",
    "Backlit Keyboard": "REMOVE",
    "Fingerprint scanner": "KEEP",

    # ============================================================
    # Remaining original predictors
    # ============================================================

    "Power Supply": "REMOVE",
    "Refresh Rate": "REMOVE",
    "Brightness": "REMOVE",
    "Graphics Memory": "REMOVE",
    "Aspect Ratio": "REMOVE",
    "Battery Life": "REMOVE",
    "USB Type C": "REMOVE",
}

In [13]:
predictor_columns = [
    col for col in df.columns
    if col != "Price (Rs)"
]

print("Original predictor count:", len(predictor_columns))
print("Decision dictionary count:", len(feature_decisions))

Original predictor count: 66
Decision dictionary count: 66


In [14]:
for decision in ["KEEP", "TRANSFORM", "ENGINEER", "REMOVE"]:

    selected = [
        col
        for col, value in feature_decisions.items()
        if value == decision
    ]

    print(f"\n{decision} ({len(selected)}):")

    for col in selected:
        print(" -", col)


KEEP (14):
 - Brand
 - Series
 - Thickness
 - Weight
 - Operating System
 - Display Size
 - Display Touchscreen
 - Processor
 - Graphic Processor
 - RAM Type
 - SSD Capacity
 - HDD Capacity
 - Battery Capacity
 - Fingerprint scanner

TRANSFORM (1):
 - Capacity

ENGINEER (0):

REMOVE (51):
 - Model
 - Warranty
 - Sales Package
 - market_status
 - Dimensions (WxDxH)
 - Colors
 - Operating System Type
 - Display Resolution
 - Pixel Density
 - Display Type
 - Display Features
 - Clock-speed
 - Chipset
 - Cache
 - Number of Cores
 - Graphics Memory
 - RAM Speed
 - Memory Slots
 - Memory Layout
 - Expandable Memory
 - SSD Type
 - SSD Interface
 - HDD Speed(RPM)
 - HDD Type
 - Battery Cell
 - Battery Type
 - Battery Life
 - Wireless LAN
 - Wi-Fi Version
 - Bluetooth
 - Bluetooth Version
 - HDMI Ports
 - USB 2.0 slots
 - USB 3.0 slots
 - USB Type C
 - Ethernet ports
 - SD Card Reader
 - Headphone Jack
 - Microphone Jack
 - Web-cam
 - Video Recording
 - Speakers
 - In-built Microphone
 - Micro

In [15]:
df["Capacity"]


0        16 GB
1        16 GB
2         8 GB
3         8 GB
4         4 GB
5         8 GB
6         8 GB
7         4 GB
8        16 GB
9         8 GB
10        8 GB
11        4 GB
12        8 GB
13        8 GB
14        4 GB
15        8 GB
16        8 GB
17        8 GB
18        8 GB
19       16 GB
20        8 GB
21        8 GB
22        8 GB
23       16 GB
24       16 GB
25        8 GB
26        4 GB
27        8 GB
28        8 GB
29        8 GB
30        8 GB
31        8 GB
32        8 GB
33        8 GB
34        8 GB
35        4 GB
36        8 GB
37        8 GB
38        8 GB
39        4 GB
40        8 GB
41        8 GB
42       16 GB
43        8 GB
44        4 GB
45        8 GB
46       16 GB
47        8 GB
48        8 GB
49       16 GB
50       16 GB
51        8 GB
52        8 GB
53        8 GB
54        8 GB
55       16 GB
56        8 GB
57        8 GB
58        8 GB
59       16 GB
60        4 GB
61       16 GB
62        8 GB
63        4 GB
64        4 GB
65        8 GB
66       1

In [16]:
df["RAM_Capacity_GB"] = (
    df["Capacity"]
    .str.extract(r"([\d.]+)", expand=False)
    .astype(float)
)

In [17]:
df["RAM_Capacity_GB"]

0        16.0
1        16.0
2         8.0
3         8.0
4         4.0
5         8.0
6         8.0
7         4.0
8        16.0
9         8.0
10        8.0
11        4.0
12        8.0
13        8.0
14        4.0
15        8.0
16        8.0
17        8.0
18        8.0
19       16.0
20        8.0
21        8.0
22        8.0
23       16.0
24       16.0
25        8.0
26        4.0
27        8.0
28        8.0
29        8.0
30        8.0
31        8.0
32        8.0
33        8.0
34        8.0
35        4.0
36        8.0
37        8.0
38        8.0
39        4.0
40        8.0
41        8.0
42       16.0
43        8.0
44        4.0
45        8.0
46       16.0
47        8.0
48        8.0
49       16.0
50       16.0
51        8.0
52        8.0
53        8.0
54        8.0
55       16.0
56        8.0
57        8.0
58        8.0
59       16.0
60        4.0
61       16.0
62        8.0
63        4.0
64        4.0
65        8.0
66       16.0
67        8.0
68        8.0
69        8.0
70       16.0
71    

In [18]:
final_features = [
    "Brand",
    "Series",
    "Thickness",
    "Weight",
    "Operating System",
    "Display Size",
    "Display Touchscreen",
    "Processor",
    "Graphic Processor",
    "RAM_Capacity_GB",
    "RAM Type",
    "SSD Capacity",
    "HDD Capacity",
    "Battery Capacity",
    "Fingerprint scanner"
]

target = "Price (Rs)"

In [19]:
df_ = df[final_features + [target]].copy()

In [20]:
print("Final df shape:", df_.shape)

Final df shape: (8176, 16)


In [21]:
print("\nFinal columns:")

for i, col in enumerate(df_.columns, start=1):
    print(f"{i}. {col}")


Final columns:
1. Brand
2. Series
3. Thickness
4. Weight
5. Operating System
6. Display Size
7. Display Touchscreen
8. Processor
9. Graphic Processor
10. RAM_Capacity_GB
11. RAM Type
12. SSD Capacity
13. HDD Capacity
14. Battery Capacity
15. Fingerprint scanner
16. Price (Rs)


In [22]:
missing_report = (
    df_.isna()
    .sum()
    .to_frame("Missing Count")
)

missing_report["Missing %"] = (
    missing_report["Missing Count"]
    / len(df_)
    * 100
)

print(missing_report)

                     Missing Count  Missing %
Brand                            0   0.000000
Series                        1090  13.331703
Thickness                      678   8.292564
Weight                         306   3.742661
Operating System                 0   0.000000
Display Size                     3   0.036693
Display Touchscreen             82   1.002935
Processor                        0   0.000000
Graphic Processor               92   1.125245
RAM_Capacity_GB                  0   0.000000
RAM Type                       170   2.079256
SSD Capacity                  1333  16.303816
HDD Capacity                  6323  77.336106
Battery Capacity              5732  70.107632
Fingerprint scanner           2674  32.705479
Price (Rs)                       0   0.000000


In [23]:
numeric_features = df_[final_features].select_dtypes(
    include="number"
).columns.tolist()

categorical_features = df_[final_features].select_dtypes(
    exclude="number"
).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nNumerical count:", len(numeric_features))

print("\nCategorical features:")
print(categorical_features)

print("\nCategorical count:", len(categorical_features))

Numerical features:
['Thickness', 'Weight', 'Display Size', 'RAM_Capacity_GB', 'SSD Capacity', 'HDD Capacity', 'Battery Capacity']

Numerical count: 7

Categorical features:
['Brand', 'Series', 'Operating System', 'Display Touchscreen', 'Processor', 'Graphic Processor', 'RAM Type', 'Fingerprint scanner']

Categorical count: 8


In [39]:
df_.to_csv("../data/processed/laptop_selected.csv", index=False)

In [40]:
print("=" * 60)
print("FINAL VALIDATION")
print("=" * 60)

print("Rows:", df_.shape[0])
print("Columns:", df_.shape[1])

print("\nPredictor count:", len(final_features))
print("Target:", target)

print("\nTarget missing values:",
      df_[target].isna().sum())

print("Duplicate rows:",
      df_.duplicated().sum())

print("\nNumerical feature count:",
      len(numeric_features))

print("Categorical feature count:",
      len(categorical_features))

print("\nFinal columns:")
print(df_.columns.tolist())

print("=" * 60)

FINAL VALIDATION
Rows: 8078
Columns: 16

Predictor count: 15
Target: Price (Rs)

Target missing values: 0
Duplicate rows: 0

Numerical feature count: 7
Categorical feature count: 8

Final columns:
['Brand', 'Series', 'Thickness', 'Weight', 'Operating System', 'Display Size', 'Display Touchscreen', 'Processor', 'Graphic Processor', 'RAM_Capacity_GB', 'RAM Type', 'SSD Capacity', 'HDD Capacity', 'Battery Capacity', 'Fingerprint scanner', 'Price (Rs)']


In [41]:
df_.duplicated().sum()

np.int64(0)

In [42]:
duplicate_rows = df_[df_.duplicated(keep=False)]

print("Rows involved in exact duplicates:",len(duplicate_rows))

duplicate_rows.head(20)

Rows involved in exact duplicates: 0


,Brand,Series,Thickness,Weight,Operating System,Display Size,Display Touchscreen,Processor,Graphic Processor,RAM_Capacity_GB,RAM Type,SSD Capacity,HDD Capacity,Battery Capacity,Fingerprint scanner,Price (Rs)


In [47]:
duplicates = df_[df_.duplicated(keep=False)]

print("Rows involved in exact duplicates:", len(duplicates))
print("Duplicate extras:", df_.duplicated().sum())

duplicates.sort_values(
    by=_.columns.tolist()
).head(30)

Rows involved in exact duplicates: 0
Duplicate extras: 0


,Brand,Series,Thickness,Weight,Operating System,Display Size,Display Touchscreen,Processor,Graphic Processor,RAM_Capacity_GB,RAM Type,SSD Capacity,HDD Capacity,Battery Capacity,Fingerprint scanner,Price (Rs)


In [45]:
df_ = df_.drop_duplicates()

In [46]:
print("Shape after removing duplicates:", df_.shape)
print("Remaining duplicates:", df_.duplicated().sum())

Shape after removing duplicates: (8078, 16)
Remaining duplicates: 0
